# Kural — Tamil Speech Emotion (EmoTa / TamilSER-DB)

Fine-tunes an emotion classifier on the Whisper encoder using **EmoTa** (936 acted Tamil
clips, 22 speakers, 5 emotions). Reports a proper **speaker-independent** Tamil score
(held-out speakers). Optionally augments with public English acted sets.

**GPU runtime** (Kaggle: T4 + Internet ON), token via Kaggle/Colab Secret `HF_TOKEN`.
5 classes: `angry, fear, happy, neutral, sad`.

> EmoTa needs approval — keep your Kaggle dataset **private**. Don't commit the audio.

In [ ]:
!pip -q install --upgrade "transformers>=4.44" "datasets[audio]>=2.20" "accelerate>=0.33" scikit-learn librosa soundfile huggingface_hub

In [ ]:
# ===== CONFIG =====
BACKBONE     = "openai/whisper-small"
HF_USERNAME  = "Venky0411"
HUB_REPO     = f"{HF_USERNAME}/whisper-small-ta-emotion"
LABELS       = ["angry", "fear", "happy", "neutral", "sad"]
label2id     = {l: i for i, l in enumerate(LABELS)}
id2label     = {i: l for l, i in label2id.items()}

EMOTA_DIR        = None   # None = auto-detect (Kaggle input / ./TamilSER-DB)
N_TEST_SPEAKERS  = 4      # held-out speakers for the speaker-independent Tamil test
USE_ENGLISH_AUG  = False  # True = add RAVDESS/CREMA-D/TESS for extra volume (domain shift)
FREEZE_ENCODER   = True
MAX_PER_SOURCE   = None
EPOCHS           = 12
BATCH_SIZE       = 16
LR               = 5e-4

In [ ]:
from huggingface_hub import login
import os
HF_TOKEN = None
try:  # Colab
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
if not HF_TOKEN:
    try:  # Kaggle — add a Secret named HF_TOKEN (Add-ons -> Secrets)
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass('HF write token: ')
login(token=HF_TOKEN)

In [ ]:
# ===== EmoTa loader (filename: SPEAKER_UTTERANCE_emotion.wav) =====
import os, glob
from datasets import Dataset, Audio, load_dataset, concatenate_datasets

EMOTA_CODE = {"ang": "angry", "fea": "fear", "hap": "happy", "neu": "neutral", "sad": "sad"}

def find_emota():
    cands = glob.glob("/kaggle/input/*/") + glob.glob("/kaggle/input/*/*/") + \
            ["/content/TamilSER-DB", "TamilSER-DB", "./TamilSER-DB"]
    for d in cands:
        if glob.glob(os.path.join(d, "*_*_*.wav")):
            return d
    raise FileNotFoundError("EmoTa/TamilSER-DB not found. On Kaggle add it as a (private) Dataset.")

def load_emota(emota_dir=None):
    emota_dir = emota_dir or find_emota()
    files = sorted(glob.glob(os.path.join(emota_dir, "*.wav")))
    rows = {"audio": [], "emo": [], "speaker": [], "source": []}
    for f in files:
        parts = os.path.basename(f)[:-4].split("_")
        if len(parts) < 3:
            continue
        emo = EMOTA_CODE.get(parts[2].lower())
        if emo is None:
            continue
        rows["audio"].append(f); rows["emo"].append(emo)
        rows["speaker"].append(parts[0]); rows["source"].append("emota")
    ds = Dataset.from_dict(rows).cast_column("audio", Audio(sampling_rate=16000))
    print(f"EmoTa: {len(ds)} clips, {len(set(rows['speaker']))} speakers, dir={emota_dir}")
    return ds

In [ ]:
# ===== optional English augmentation (label normalisation + defensive loader) =====
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
EMO_MAP = {
    "neutral": "neutral", "calm": "neutral", "normal": "neutral",
    "happy": "happy", "happiness": "happy", "joy": "happy",
    "sad": "sad", "sadness": "sad",
    "angry": "angry", "anger": "angry", "mad": "angry",
    "fear": "fear", "fearful": "fear", "fright": "fear",
    "disgust": None, "disgusted": None,
    "surprise": None, "surprised": None, "ps": None, "pleasant surprise": None,
}
def canon(v):
    return None if v is None else EMO_MAP.get(str(v).strip().lower().replace("_", " "))

def load_norm(name, source, config=None, revision=None, split="train"):
    ds = load_dataset(name, config, split=split, revision=revision, verification_mode="no_checks")
    cands = [c for c in ["label", "labels", "emotion", "emotions", "class"] if c in ds.column_names]
    label_col = next((c for c in cands if ds.features[c].__class__.__name__ == "ClassLabel"),
                     cands[0] if cands else None)
    feat = ds.features[label_col] if label_col else None
    is_cl = feat.__class__.__name__ == "ClassLabel" if feat is not None else False
    def _m(b):
        v = b[label_col] if label_col else None
        if is_cl and isinstance(v, int):
            v = feat.int2str(v)
        b["emo"] = canon(v); b["source"] = source
        return b
    ds = ds.map(_m)
    ds = ds.filter(lambda b: b["emo"] is not None)
    ds = ds.cast_column("audio", Audio(sampling_rate=16000)).select_columns(["audio", "emo", "source"])
    return ds.select(range(min(MAX_PER_SOURCE, len(ds)))) if MAX_PER_SOURCE else ds

ENGLISH = [("confit/cremad-parquet", "cremad", None, None),
           ("confit/ravdess-parquet", "ravdess", "fold1", None),
           ("AbstractTTS/TESS", "tess", None, None)]

In [ ]:
# ===== load EmoTa (required) + optional English =====
emota = load_emota(EMOTA_DIR)

english = None
if USE_ENGLISH_AUG:
    parts = []
    for name, src, cfg, rev in ENGLISH:
        try:
            d = load_norm(name, src, config=cfg, revision=rev)
            print(f"  OK  {src:8s} {len(d):6d} rows"); parts.append(d)
        except Exception as e:
            print(f"  SKIP {src}: {type(e).__name__}: {str(e)[:80]}")
    english = concatenate_datasets(parts) if parts else None

In [ ]:
# ===== speaker-independent split: hold out N speakers from EmoTa for TEST =====
from collections import Counter

spks = sorted(set(emota["speaker"]))
test_spks = set(spks[-N_TEST_SPEAKERS:])
print("held-out test speakers:", sorted(test_spks))

emota_test  = emota.filter(lambda b: b["speaker"] in test_spks).remove_columns("speaker")
emota_train = emota.filter(lambda b: b["speaker"] not in test_spks).remove_columns("speaker")

train_parts = ([english] if (USE_ENGLISH_AUG and english is not None) else []) + [emota_train]
train_ds = concatenate_datasets(train_parts).shuffle(seed=42)
tamil_test = emota_test
print("train:", len(train_ds), "| tamil_test (held-out speakers):", len(tamil_test))
print("train labels:", dict(Counter(train_ds["emo"])))
print("test  labels:", dict(Counter(tamil_test["emo"])))

In [ ]:
# ===== feature extraction (Whisper log-mel) + label ids =====
from transformers import WhisperFeatureExtractor
fe = WhisperFeatureExtractor.from_pretrained(BACKBONE)

def prep(b):
    b["input_features"] = fe(b["audio"]["array"], sampling_rate=16000).input_features[0]
    b["labels"] = label2id[b["emo"]]
    return b

train_ds = train_ds.map(prep, remove_columns=train_ds.column_names, num_proc=2)
tamil_test = tamil_test.map(prep, remove_columns=tamil_test.column_names, num_proc=2)

In [ ]:
# ===== model: Whisper encoder + classification head =====
from transformers import WhisperForAudioClassification
model = WhisperForAudioClassification.from_pretrained(
    BACKBONE, num_labels=len(LABELS), label2id=label2id, id2label=id2label)
if FREEZE_ENCODER:
    for p in model.encoder.parameters():
        p.requires_grad = False
print(f"trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "f1_macro": f1_score(p.label_ids, preds, average="macro")}

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

args = TrainingArguments(
    output_dir="./whisper-ta-emotion",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to=["none"],
    push_to_hub=True,
    hub_model_id=HUB_REPO,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=tamil_test,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# ===== speaker-independent Tamil-test report =====
from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(tamil_test)
y, yhat = pred.label_ids, pred.predictions.argmax(1)
print(classification_report(y, yhat, target_names=LABELS, digits=3))
print("confusion matrix (rows=true):\n", confusion_matrix(y, yhat))

In [ ]:
# ===== push model + feature extractor =====
fe.push_to_hub(HUB_REPO)
trainer.push_to_hub(**{"dataset": "EmoTa (TamilSER-DB)", "language": "ta",
                       "tasks": "audio-classification", "finetuned_from": BACKBONE})
print("Pushed:", HUB_REPO)

In [ ]:
# ===== quick inference demo =====
import torch, librosa
def predict_emotion(path):
    y, _ = librosa.load(path, sr=16000, mono=True)
    feats = fe(y, sampling_rate=16000, return_tensors="pt").input_features.to(model.device)
    with torch.no_grad():
        probs = model(feats).logits.softmax(-1)[0]
    i = int(probs.argmax())
    return id2label[i], float(probs[i])

# example: predict_emotion('/kaggle/input/<your-emota>/01_01_ang.wav')